# HDNNP2nd Model Showcase: Symmetry Functions and Dipole Prediction

This notebook demonstrates the **HDNNP2nd** (2nd generation High-Dimensional Neural Network
Potential) model in kgcnn_torch, using the **FreeSolv** dataset (642 small molecules). The Behler-Parrinello HDNNP uses:

- **Atom-centered symmetry functions (ACSF)** as descriptors:
  - Radial (G2-type): Gaussian-weighted pair distances
  - Angular (G4-type): Angle-dependent triplet terms
- **Element-specific neural networks** (RelationalMLP): Different weight matrices for each element type
- **Sum pooling** of per-atom energies to get total molecular energy
- Optional **dipole prediction** from partial charges

Reference: Behler, J. Chem. Phys. 134, 074106 (2011)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Loading the FreeSolv Dataset

We use the FreeSolv dataset (642 molecules with hydration free energies) following
the Keras notebook. The dataset is loaded via `FreeSolvDataset`, then post-processed to add:
1. Range-based edges within a 5.0 Å cutoff (replacing bond-graph edges)
2. Angle triplet indices for angular symmetry functions
3. Zero dipole and charge targets (as in the Keras reference)

In [ ]:
from kgcnn_torch.data.datasets.FreeSolvDataset import FreeSolvDataset
from collections import defaultdict

dataset = FreeSolvDataset()
print(f"Loaded {len(dataset)} molecules from FreeSolv")


def add_range_and_angles(data, cutoff=5.0):
    """Add range-based edges and angle indices for HDNNP2nd."""
    pos = data.pos
    n = pos.size(0)

    # Compute all pairwise distances and build range edges within cutoff
    diff = pos.unsqueeze(0) - pos.unsqueeze(1)
    dist = (diff * diff).sum(dim=-1).sqrt()
    mask = (dist < cutoff) & (dist > 1e-6)
    src, tgt = torch.where(mask)
    data.edge_index = torch.stack([src, tgt], dim=0)

    # Build angle indices: for each center atom, enumerate pairs of neighbors
    neighbors = defaultdict(list)
    for s, t in zip(src.tolist(), tgt.tolist()):
        neighbors[t].append(s)

    angle_i, angle_j, angle_k = [], [], []
    for center, nbrs in neighbors.items():
        for a in range(len(nbrs)):
            for b in range(a + 1, len(nbrs)):
                angle_i.append(center)
                angle_j.append(nbrs[a])
                angle_k.append(nbrs[b])

    if angle_i:
        data.angle_index = torch.tensor([angle_i, angle_j, angle_k], dtype=torch.long)
    else:
        data.angle_index = torch.zeros((3, 0), dtype=torch.long)

    # Add zero dipole and charge targets (matching Keras notebook)
    data.dipole = torch.zeros(1, 3, dtype=torch.float32)
    data.total_charge = torch.zeros(1, 1, dtype=torch.float32)

    return data


# Process all molecules
pyg_list = []
skipped = 0
for i in range(len(dataset)):
    data = dataset[i].clone()
    if data.pos is None or data.z is None:
        skipped += 1
        continue
    data = add_range_and_angles(data, cutoff=5.0)
    if data.angle_index.size(1) > 0:
        pyg_list.append(data)
    else:
        skipped += 1

# Detect element types
all_z = torch.cat([d.z for d in pyg_list])
element_types = sorted(all_z.unique().tolist())

print(f"Prepared {len(pyg_list)} molecules ({skipped} skipped)")
print(f"Elements found: {element_types}")
print(f"Sample molecule:")
print(f"  z = {pyg_list[0].z[:10].numpy()} ...")
print(f"  edge_index shape = {pyg_list[0].edge_index.shape}")
print(f"  angle_index shape = {pyg_list[0].angle_index.shape}")
print(f"  energy (y) = {pyg_list[0].y.numpy()}")

## 2. Behler G2/G4 Symmetry Functions

The HDNNP2nd model uses atom-centered symmetry functions (ACSF) as molecular descriptors.

**G2 (Radial):** Gaussian-weighted pairwise distances
$$G_i^2 = \sum_j e^{-\eta(r_{ij} - R_s)^2} \cdot f_c(r_{ij})$$

**G4 (Angular):** Three-body angular terms
$$G_i^4 = 2^{1-\zeta} \sum_{j,k} (1 + \lambda\cos\theta_{ijk})^\zeta \cdot e^{-\eta(r_{ij}^2 + r_{ik}^2 + r_{jk}^2)} \cdot f_c(r_{ij}) f_c(r_{ik}) f_c(r_{jk})$$

The cosine cutoff function ensures smooth decay to zero:
$$f_c(r) = \begin{cases} \frac{1}{2}(\cos(\pi r / R_c) + 1) & r < R_c \\ 0 & r \geq R_c \end{cases}$$

In [ ]:
from kgcnn_torch.models.hdnnp2nd import HDNNP2ndBehlerModel

# Configure G2 radial symmetry functions
g2_eta = [0.0, 0.3, 1.0, 4.0]    # Width parameter
g2_rs = [0.0, 1.0, 2.0, 3.0]     # Shift parameter
g2_rc = 6.0                        # Cutoff radius

# Configure G4 angular symmetry functions
g4_eta = [0.0, 0.3]               # Width parameter
g4_zeta = [1.0, 4.0]              # Angular resolution
g4_lamda = [-1.0, 1.0]            # Angular direction (+1 or -1)
g4_rc = 6.0                        # Cutoff radius

print("G2 symmetry function parameters:")
print(f"  eta = {g2_eta}")
print(f"  Rs = {g2_rs}")
print(f"  Rc = {g2_rc}")
print(f"  Number of G2 features per element: {len(g2_eta) * len(g2_rs)}")

print("\nG4 symmetry function parameters:")
print(f"  eta = {g4_eta}")
print(f"  zeta = {g4_zeta}")
print(f"  lambda = {g4_lamda}")
print(f"  Rc = {g4_rc}")
print(f"  Number of G4 features per element pair: {len(g4_eta) * len(g4_zeta) * len(g4_lamda)}")

## 3. Build HDNNP2nd Behler Model

The model architecture:
1. Compute G2 + G4 descriptors for each atom
2. Concatenate into a per-atom feature vector
3. Apply element-specific MLP (RelationalMLP) to produce per-atom energies
4. Sum per-atom energies to get total molecular energy

In [ ]:
model_behler = HDNNP2ndBehlerModel(
    element_types=element_types,
    g2_eta=g2_eta,
    g2_rs=g2_rs,
    g2_rc=g2_rc,
    g4_eta=g4_eta,
    g4_zeta=g4_zeta,
    g4_lamda=g4_lamda,
    g4_rc=g4_rc,
    relational_units=[128, 128, 128, 1],
    relational_activation=["swish", "swish", "swish", "linear"],
    num_relations=96,
    node_pooling="sum",
    output_embedding="graph",
    num_targets=1,
)

print(f"HDNNP2nd Behler model parameters: {sum(p.numel() for p in model_behler.parameters()):,}")
print(f"Number of G2 features: {model_behler.n_g2}")
print(f"Number of G4 features: {model_behler.n_g4}")
print(f"Total descriptor dimension: {model_behler.n_g2 + model_behler.n_g4}")

In [ ]:
# Test forward pass
from torch_geometric.loader import DataLoader

test_loader = DataLoader(pyg_list[:4], batch_size=4, shuffle=False)
test_batch = next(iter(test_loader))

model_behler.eval()
with torch.no_grad():
    output = model_behler(test_batch)
print(f"Output shape: {output.shape}")
print(f"Predictions: {output.flatten().numpy()}")

## 4. HDNNP2nd with Weighted ACSF (wACSF)

The weighted variant uses atomic-number weighting in the symmetry functions,
making them element-aware without separate channels per element pair.

In [ ]:
from kgcnn_torch.models.hdnnp2nd import HDNNP2ndModel

model_wacsf = HDNNP2ndModel(
    element_types=element_types,
    n_rad_features=22,
    n_ang_features=10,
    cutoff=8.0,
    num_relations=96,
    relational_units=[128, 128, 128, 1],
    relational_activation=["swish", "swish", "swish", "linear"],
    use_batch_norm=False,
    node_pooling="sum",
    use_output_mlp=False,
    num_targets=1,
    output_embedding="graph",
)

print(f"HDNNP2nd wACSF model parameters: {sum(p.numel() for p in model_wacsf.parameters()):,}")
print(f"Descriptor dimension: {model_wacsf.n_rad_features + model_wacsf.n_ang_features}")

## 5. HDNNP2nd with Dipole Prediction

When `predict_dipole=True`, the model also predicts molecular dipole moments
from partial atomic charges:

$$\vec{\mu} = \sum_i q_i \vec{r}_i$$

The model outputs:
- Energy (scalar per molecule)
- Dipole moment (3D vector per molecule)
- Total charge (scalar per molecule, when `has_charge_input=False`)

In [ ]:
model_dipole = HDNNP2ndModel(
    element_types=element_types,
    n_rad_features=22,
    n_ang_features=10,
    cutoff=8.0,
    num_relations=96,
    relational_units=[128, 128, 128, 1],
    relational_activation=["swish", "swish", "swish", "linear"],
    node_pooling="sum",
    use_output_mlp=False,
    num_targets=1,
    output_embedding="graph",
    predict_dipole=True,
    has_charge_input=False,
)

print("Model with dipole prediction:")
print(f"  Parameters: {sum(p.numel() for p in model_dipole.parameters()):,}")

In [ ]:
# Test dipole prediction
model_dipole.eval()
with torch.no_grad():
    output = model_dipole(test_batch)

# Output is a tuple: (energy, dipole, total_charge)
energy_out, dipole_out, charge_out = output
print(f"Energy shape: {energy_out.shape}")
print(f"Dipole shape: {dipole_out.shape}")
print(f"Total charge shape: {charge_out.shape}")
print(f"\nEnergy: {energy_out.flatten().numpy()}")
print(f"Dipole: {dipole_out.numpy()}")
print(f"Total charge: {charge_out.flatten().numpy()}")

## 6. Training: Charge as Output

Outputs of the model are energy, dipole, and total charge. No additional charge input
is needed. We train with a multi-output MAE loss on the FreeSolv dataset.

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    np.arange(len(pyg_list)), test_size=0.2, random_state=42
)
train_data = [pyg_list[i] for i in train_idx]
test_data = [pyg_list[i] for i in test_idx]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

In [ ]:
class HDNNP2ndDipoleLoss(nn.Module):
    """Multi-output loss for HDNNP2nd with dipole prediction.

    Handles both charge-as-output (3 outputs) and charge-as-input (2 outputs).
    """

    def __init__(self, energy_weight=1.0, dipole_weight=1.0, charge_weight=1.0):
        super().__init__()
        self.energy_weight = energy_weight
        self.dipole_weight = dipole_weight
        self.charge_weight = charge_weight
        self.mae = nn.L1Loss()

    def forward(self, pred, batch):
        if isinstance(pred, tuple) and len(pred) == 3:
            energy_pred, dipole_pred, charge_pred = pred
        elif isinstance(pred, tuple) and len(pred) == 2:
            energy_pred, dipole_pred = pred
            charge_pred = None
        else:
            raise ValueError(f"Expected tuple of 2 or 3, got {type(pred)}")

        energy_target = batch.y
        if energy_target.dim() == 1:
            energy_target = energy_target.unsqueeze(-1)
        loss = self.energy_weight * self.mae(energy_pred, energy_target)

        dipole_target = batch.dipole
        loss = loss + self.dipole_weight * self.mae(dipole_pred, dipole_target)

        if charge_pred is not None:
            charge_target = batch.total_charge
            if charge_target.dim() == 1:
                charge_target = charge_target.unsqueeze(-1)
            loss = loss + self.charge_weight * self.mae(charge_pred, charge_target)

        return loss

In [ ]:
import time
from datetime import timedelta

# Create a fresh model for training
model_dipole = HDNNP2ndModel(
    element_types=element_types,
    n_rad_features=22,
    n_ang_features=10,
    cutoff=8.0,
    num_relations=96,
    relational_units=[128, 128, 128, 1],
    relational_activation=["swish", "swish", "swish", "linear"],
    node_pooling="sum",
    use_output_mlp=False,
    num_targets=1,
    output_embedding="graph",
    predict_dipole=True,
    has_charge_input=False,
)
model_dipole = model_dipole.to(device)

optimizer = torch.optim.Adam(model_dipole.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.01, total_iters=300
)
loss_fn = HDNNP2ndDipoleLoss(energy_weight=1.0, dipole_weight=1.0, charge_weight=1.0)

history_charge_output = {"train_loss": [], "val_loss": []}

start = time.process_time()
for epoch in range(300):
    model_dipole.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model_dipole(batch)
        loss = loss_fn(pred, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    history_charge_output["train_loss"].append(epoch_loss / max(n_batches, 1))

    model_dipole.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model_dipole(batch)
            loss = loss_fn(pred, batch)
            val_loss += loss.item()
            n_val += 1
    history_charge_output["val_loss"].append(val_loss / max(n_val, 1))

    scheduler.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/300 - train_loss: {history_charge_output['train_loss'][-1]:.4f} "
              f"- val_loss: {history_charge_output['val_loss'][-1]:.4f}")

stop = time.process_time()
print(f"\nTraining time: {timedelta(seconds=stop - start)}")

## 7. Training Curves (Charge as Output)

In [ ]:
from kgcnn_torch.utils.plots import plot_train_test_loss

plot_train_test_loss(
    [history_charge_output],
    loss_name="train_loss",
    val_loss_name="val_loss",
    model_name="HDNNP2nd",
    data_unit="kcal/mol",
    dataset_name="FreeSolv",
);

## 8. Evaluate Predictions (Charge as Output)

In [ ]:
import matplotlib.pyplot as plt

model_dipole.eval()
all_energy_pred, all_energy_true = [], []
all_dipole_pred, all_dipole_true = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        energy_pred, dipole_pred, charge_pred = model_dipole(batch)
        all_energy_pred.append(energy_pred.cpu().numpy())
        all_energy_true.append(batch.y.cpu().numpy())
        all_dipole_pred.append(dipole_pred.cpu().numpy())
        all_dipole_true.append(batch.dipole.cpu().numpy())

energy_pred_arr = np.concatenate(all_energy_pred).flatten()
energy_true_arr = np.concatenate(all_energy_true).flatten()
dipole_pred_arr = np.concatenate(all_dipole_pred)
dipole_true_arr = np.concatenate(all_dipole_true)

print(f"Energy MAE: {np.mean(np.abs(energy_pred_arr - energy_true_arr)):.4f} kcal/mol")
print(f"Dipole MAE: {np.mean(np.abs(dipole_pred_arr - dipole_true_arr)):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(energy_true_arr, energy_pred_arr, alpha=0.4, s=10)
lims = [min(energy_true_arr.min(), energy_pred_arr.min()),
        max(energy_true_arr.max(), energy_pred_arr.max())]
ax1.plot(lims, lims, 'r-')
ax1.set_xlabel("True Energy (kcal/mol)")
ax1.set_ylabel("Predicted Energy (kcal/mol)")
ax1.set_title("Energy Prediction")

ax2.scatter(dipole_true_arr.flatten(), dipole_pred_arr.flatten(), alpha=0.3, s=5)
lims = [min(dipole_true_arr.min(), dipole_pred_arr.min()),
        max(dipole_true_arr.max(), dipole_pred_arr.max())]
ax2.plot(lims, lims, 'r-')
ax2.set_xlabel("True Dipole Component")
ax2.set_ylabel("Predicted Dipole Component")
ax2.set_title("Dipole Prediction")

plt.tight_layout()
plt.show()

## 9. Training: Charge as Input

When `has_charge_input=True`, the model takes the total molecular charge as input
and uses it to correct partial charges via charge conservation:

$$q_i^{\text{corrected}} = q_i^{\text{raw}} + \frac{Q_{\text{total}} - \sum_j q_j^{\text{raw}}}{N_{\text{atoms}}}$$

The outputs are then (energy, dipole) -- total charge is no longer predicted.
We transfer weights from the charge-as-output model and fine-tune.

In [ ]:
# Create charge-as-input model
model_charge_input = HDNNP2ndModel(
    element_types=element_types,
    n_rad_features=22,
    n_ang_features=10,
    cutoff=8.0,
    num_relations=96,
    relational_units=[128, 128, 128, 1],
    relational_activation=["swish", "swish", "swish", "linear"],
    node_pooling="sum",
    use_output_mlp=False,
    num_targets=1,
    output_embedding="graph",
    predict_dipole=True,
    has_charge_input=True,
)

# Transfer compatible weights from charge-as-output model
state_src = model_dipole.state_dict()
state_dst = model_charge_input.state_dict()
transferred = 0
for key in state_dst:
    if key in state_src and state_src[key].shape == state_dst[key].shape:
        state_dst[key] = state_src[key]
        transferred += 1
model_charge_input.load_state_dict(state_dst)
print(f"Transferred {transferred}/{len(state_dst)} parameter tensors")

model_charge_input = model_charge_input.to(device)
optimizer = torch.optim.Adam(model_charge_input.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.01, total_iters=300
)
loss_fn_ci = HDNNP2ndDipoleLoss(energy_weight=1.0, dipole_weight=1.0, charge_weight=1.0)

history_charge_input = {"train_loss": [], "val_loss": []}

start = time.process_time()
for epoch in range(300):
    model_charge_input.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model_charge_input(batch)
        loss = loss_fn_ci(pred, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    history_charge_input["train_loss"].append(epoch_loss / max(n_batches, 1))

    model_charge_input.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model_charge_input(batch)
            loss = loss_fn_ci(pred, batch)
            val_loss += loss.item()
            n_val += 1
    history_charge_input["val_loss"].append(val_loss / max(n_val, 1))

    scheduler.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/300 - train_loss: {history_charge_input['train_loss'][-1]:.4f} "
              f"- val_loss: {history_charge_input['val_loss'][-1]:.4f}")

stop = time.process_time()
print(f"\nTraining time: {timedelta(seconds=stop - start)}")

In [ ]:
# Plot training curves for both variants
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_charge_output["train_loss"], label="Train", alpha=0.8)
ax1.plot(history_charge_output["val_loss"], label="Val", alpha=0.8)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Charge as Output")
ax1.legend()

ax2.plot(history_charge_input["train_loss"], label="Train", alpha=0.8)
ax2.plot(history_charge_input["val_loss"], label="Val", alpha=0.8)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Charge as Input")
ax2.legend()

plt.suptitle("HDNNP2nd Training on FreeSolv", fontsize=14)
plt.tight_layout()
plt.show()

# Evaluate charge-as-input model
model_charge_input.eval()
all_energy_pred, all_energy_true = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        energy_pred, dipole_pred = model_charge_input(batch)
        all_energy_pred.append(energy_pred.cpu().numpy())
        all_energy_true.append(batch.y.cpu().numpy())

ep = np.concatenate(all_energy_pred).flatten()
et = np.concatenate(all_energy_true).flatten()
print(f"Charge-as-input Energy MAE: {np.mean(np.abs(ep - et)):.4f} kcal/mol")

## Summary

This notebook demonstrated the HDNNP2nd model family in kgcnn_torch on the **FreeSolv** dataset (642 molecules):

1. **Behler G2/G4 symmetry functions**: Fixed-parameter radial and angular descriptors
2. **Weighted ACSF (wACSF)**: Element-weighted symmetry functions with per-type parameters
3. **Element-specific MLPs**: RelationalMLP applies different weights per element type
4. **Dipole prediction**: From partial atomic charges via charge-weighted positions
5. **Charge as output**: Model predicts energy, dipole, and total charge (3 outputs)
6. **Charge as input**: Model takes total charge as input for charge conservation (2 outputs)
7. **Weight transfer**: Shared weights transferred between model variants